# A.X-3.1-Light 자체 라벨링 (SKT 라우터 챌린지)

**런타임 → 런타임 유형 변경 → GPU: L4 또는 A100** (T4 16GB는 bf16 7B가 안 올라감).

순서: ① 번들 업로드 → ② vLLM 설치 → ③ pilot(공개 1,951문항 재현, 주최측 라벨과 일치율) → ④ pool(신규 ~7k문항 라벨) → ⑤ 결과 zip 다운로드.
중간에 끊겨도 같은 셀을 다시 실행하면 이어서 진행됩니다(재개 지원). 전체 예상: L4 기준 pilot 10~15분, pool 40~70분.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## ① 번들 업로드
로컬에서 만든 `router_label_bundle.zip`을 올립니다. Drive에 두면(`MyDrive/router_label_bundle.zip`) 자동으로 사용합니다.

In [ ]:
import os, glob
BUNDLE = None
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    cand = '/content/drive/MyDrive/router_label_bundle.zip'
    if os.path.exists(cand):
        BUNDLE = cand
except Exception as e:
    print('drive skip:', e)
if BUNDLE is None:
    from google.colab import files
    up = files.upload()  # router_label_bundle.zip 선택
    BUNDLE = next(iter(up))
!rm -rf /content/label && mkdir -p /content/label && unzip -q -o "$BUNDLE" -d /content/label
%cd /content/label/colab-label
!ls -la bundle && cat bundle/meta.json | head -30
# 이전 실행 결과가 Drive에 있으면 이어서 진행
if os.path.isdir('/content/drive/MyDrive/router_label_out'):
    !mkdir -p out && cp -n /content/drive/MyDrive/router_label_out/*.jsonl out/ 2>/dev/null; ls out

## ② vLLM 설치 (약 3~6분)

In [ ]:
%pip install -q vllm
import vllm, torch; print('vllm', vllm.__version__, 'torch', torch.__version__, 'gpu', torch.cuda.get_device_name(0))

## ③ Pilot — 주최측 라벨 재현 (온도 0.7 / 1.0 두 가지)
끝나면 family별 일치율 표가 출력되고 `out/pilot_report.json`에 저장됩니다.
**판정 기준**: `within.25` ≥ 0.85, `outlen corr` ≥ 0.8이면 pool 진행. 0.75 미만이면 중단하고 Claude에게 보고.

In [ ]:
!python run_labels.py --stage pilot --temps 0.7 --n 4 --max-tokens 2048 --batch 512
!python run_labels.py --stage pilot --temps 0.7 --n 4 --max-tokens 2048 --batch 512 --instruct v1 --tag _v1


## ④ Pool — 신규 문항 라벨링
pilot에서 일치율(within.25)이 높은 온도를 자동 선택합니다. 시간이 부족하면 `--limit-per-family 400` 등으로 줄여도 됩니다.

In [ ]:
import json, re
rep = json.load(open('out/pilot_report.json'))
best = max(rep.items(), key=lambda kv: kv[1]['within_025'])
m = re.match(r'labels_pilot_T([\d.]+)(_v\d)?\.jsonl', best[0])
T = float(m.group(1)); TAG = m.group(2) or ''; INS = TAG.lstrip('_')
print('pilot best', best[0], '-> temp', T, 'instruct', repr(INS), 'within.25', round(best[1]['within_025'],3), 'agree', round(best[1]['agree_bin'],3))
!python run_labels.py --stage pool --temp $T --n 4 --max-tokens 2048 --batch 512 --instruct "$INS" --tag "$TAG"


## ⑤ 결과 저장 (Drive + 다운로드)
`router_label_out.zip`을 로컬 `official-router/colab-label/`에 두고 `python colab-label/ingest_labels.py router_label_out.zip`을 실행하면 됩니다.

In [ ]:
!mkdir -p /content/drive/MyDrive/router_label_out 2>/dev/null; cp out/*.jsonl out/*.json /content/drive/MyDrive/router_label_out/ 2>/dev/null; echo drive-saved
!cd out && zip -q -r ../router_label_out.zip . && ls -la ../router_label_out.zip
from google.colab import files
files.download('/content/label/colab-label/router_label_out.zip')

### 중간 백업 (선택)
pool 진행 중 세션이 끊길 것 같으면 아래 셀로 지금까지의 결과를 Drive에 복사해 두세요. 다음 세션에서 ①을 다시 실행하면 자동으로 이어집니다.

In [ ]:
!mkdir -p /content/drive/MyDrive/router_label_out && cp out/*.jsonl /content/drive/MyDrive/router_label_out/ && ls -la /content/drive/MyDrive/router_label_out